# ReasonLite — first GPU baseline
Select **Runtime > Change runtime type > T4 GPU** (or another GPU). Run cells in order. Upload `ReasonLite_v0.1.zip` in cell 1. Do not run SFT until the baseline succeeds. No results are prefilled.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
assert 'ReasonLite_v0.1.zip' in uploaded, 'Upload the named source archive'
with zipfile.ZipFile('ReasonLite_v0.1.zip') as z:
    z.extractall('/content')
os.chdir('/content/ReasonLite')
print(Path.cwd())

In [ ]:
import sys, subprocess, os
# Use only the first visible GPU consistently.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run('-m', 'pip', 'install', '-r', 'requirements.txt')
run('-c', 'import torch; assert torch.cuda.is_available(), "Enable GPU runtime"; print(torch.__version__, torch.cuda.get_device_name())')
run('-m', 'unittest', 'discover', '-s', 'tests', '-v')

## Prepare frozen splits
Run once. The official test is stored but must not be evaluated during development.

In [ ]:
run('-m','reasonlite.data','--include-test')

## 16-question smoke run
Check GPU memory, formatting, invalid rate and truncation. If changing the generation cap, use the same new cap for all later runs.

In [ ]:
run('-m','reasonlite.evaluate','--device','cuda','--limit','16','--out','runs/e0_smoke')

## Full validation baseline (128 questions)
Run if the smoke is satisfactory. No official test evaluation today.

In [ ]:
run('-m','reasonlite.evaluate','--device','cuda','--limit','128','--out','runs/e0_validation')

## Download baseline evidence now
Preserve the data manifest and split files in private storage; do not commit the raw data. Colab temporary storage may disappear.

In [ ]:
import shutil, json
from pathlib import Path
print(Path('runs/e0_validation/metrics.json').read_text())
shutil.make_archive('/content/ReasonLite_baseline_evidence','zip','/content/ReasonLite','runs')
files.download('/content/ReasonLite_baseline_evidence.zip')
shutil.make_archive('/content/ReasonLite_frozen_data','zip','/content/ReasonLite','data')
files.download('/content/ReasonLite_frozen_data.zip')

## Optional next phase: SFT smoke (only after baseline)
These cells are starter training code, not a verified GPU run. On a fresh runtime restore the exact frozen data directory first. Use a new output directory for any rerun.

In [ ]:
run('-m','reasonlite.train_sft','--out','runs/sft_smoke','--max-steps','2')

## Actual SFT (next sprint)
Do not execute if the smoke fails.

In [ ]:
run('-m','reasonlite.train_sft','--out','runs/sft')

In [ ]:
run('-m','reasonlite.evaluate','--device','cuda','--adapter','runs/sft/adapter','--limit','128','--out','runs/e1_validation')
run('-m','reasonlite.compare','runs/e0_validation','runs/e1_validation','--out','runs/e0_vs_e1.json')

In [ ]:
shutil.make_archive('/content/ReasonLite_sft_evidence','zip','/content/ReasonLite','runs')
files.download('/content/ReasonLite_sft_evidence.zip')